In [4]:
import torch
import torch.nn as nn

## Merging Attention Heads

- This is the reverse of section 6 which had the ``view + transpose`` functions. 
- Completing the attention head we produce an ``output`` of shape ``(B, h, T, C/h)`` which needs to be merged back to shape ``(B,T,C)``.
- The sequence of operations that are done to merge are:
    1. `transpose()` - This converts `(B, h, T, C/h) -> (B, T, h, C/h)`
    2. `.contiguous()` - physically relays the data in memory so the logical order mathces the RAM order again.
    3. `.view()` - This changes how to read the data `(B, T, h, C/h) -> (B, T, C)`

### Understanding Tensors under the hood

When we say: 
- **Logical order**: This is the order in which the data is read based on the data structure (Tensor representation)
- **RAM order**: This is the order the data is presented in the RAM (an array)

No matter how many dimension a tensor is representing, it's stored in a single one dimensional array (RAM).<br> Multi-dimensionality is something that PyTorch simulates on top of the hardware. 

**How?**
- To respresent a tensor in abstracted form we (PyTorch) need(s) two peices of information to create a mapping:
  1. ``shape`` - How many elements along each axis
  2. ``strides`` - How many flat-memory positions do I jump if i increase this axis's index by 1
  3. 

**EXAMPLE**

```python
shape (2, 3):
index: 0, 1

stride[1] = 1 <--- Last dimension
stride[0] = 3

CONCLUSION
strides = (3, 1)
```

```python
shape (2, 3, 4)
index: 0, 1, 2

stride[2] = 1 <--- Last dimension
stride[1] = 4 
stride[0] = 1*3*4 = 12

CONCLUSION
stride = (12, 4, 1)
```

If we want to reach 5a and 9b we would apply the following:

```python
t[0, 1, 0] -> RAM[0*12 + 1*4 + 0*1] = RAM[4]  ✓ correct
t[1, 2, 0] -> RAM[1*12 + 2*4 + 0*1] = RAM[20] ✓ correct

LOGICAL ORDER
[
  [
    [1a, 2a, 3a, 4a], [5a, 6a, 7a, 8a], [9a, 10a, 11a, 12a]
  ],
  [
    [1b, 2b, 3b, 4b], [5b, 6b, 7b, 8b], [9b, 10b, 11b, 12b]
  ]
]
RAM ORDER
[1a, 2a, 3a, 4a, 5a, 6a, 7a, 8a, 9a, 10a, 11a, 12a, 1b, 2b, 3b, 4b, 5b, 6b, 7b, 8b, 9b, 10b, 11b, 12b] 
```


The standard indexing is known as `row-major` which the strides are `(12, 4, 1)`. So when you ask `t[0, 1, 0]`, via the row-major calculation RAM and LOGICAL are matching enumeration outputs.

#### Calling `.view()` is FREE

Suppose we change t to be of shape `(2, 12)`

```python
LOGICAL ORDER
[
  [
    [1a, 2a, 3a, 4a, 5a, 6a, 7a, 8a, 9a, 10a, 11a, 12a]
  ],
  [
    [1b, 2b, 3b, 4b, 5b, 6b, 7b, 8b, 9b, 10b, 11b, 12b]
  ]
]

RAM ORDER
[1a, 2a, 3a, 4a, 5a, 6a, 7a, 8a, 9a, 10a, 11a, 12a, 1b, 2b, 3b, 4b, 5b, 6b, 7b, 8b, 9b, 10b, 11b, 12b] 
```
So recompute the new row-major:

```python
shape (2, 12)
index: 0,  1

stride[1] = 1 <--- Last dimension
stride[0] = 1*12

CONCLUSION
stride = (12, 1) 


5a = t[0, 4] -> RAM[0*12 + 1*4] = RAM[4]  ✓ correct
9b = t[1, 8] -> RAM[1*12 + 1*8] = RAM[20] ✓ correct
```

### Calling `.transpose()` isn't FREE

Assume that on ```t.shape = (2, 3, 4)``` (strides `(12, 4, 1)`) we applied `.transpose(0, 1)`, producing `t.shape = (3, 2, 4)`

```python
shape (3, 2, 4)

LOGICAL VIEW  (new_t[i,j,k] = old_t[j,i,k])
[
  [[1a, 2a, 3a, 4a], [1b, 2b, 3b, 4b]], 
  [[5a, 6a, 7a, 8a], [5b, 6b, 7b, 8b]], 
  [[9a, 10a, 11a, 12a], [9b, 10b, 11b, 12b]], 
]

RAM VIEW (unchanged - still the original flat array, transpose moves no data)

[1a, 2a, 3a, 4a, 5a, 6a, 7a, 8a, 9a, 10a, 11a, 12a, 1b, 2b, 3b, 4b, 5b, 6b, 7b, 8b, 9b, 10b, 11b, 12b] 

CORRECT strides after transpose = swap stride[0] and stride[1] of the ORIGINAL (12,4,1)
(NOT freshly computed from the new shape):

stride[0] = 4    <--- inherited from old stride[1]
stride[1] = 12   <--- inherited from old stride[0]
stride[2] = 1    <--- untouched

CONCLUSION
strides = (4, 12, 1)

Verify indexing still gives correct values:
5b = t[1, 1, 0] -> RAM[4*1 + 12*1 + 1*0] = RAM[16] = 5b   ✓ correct
9a = t[2, 0, 0] -> RAM[4*2 + 12*0 + 1*0] = RAM[8]  = 9a   ✓ correct

So indexing a transposed tensor ALWAYS works correctly the swapped strides
make the arithmetic land on the right RAM position, every time.

CONTRAST - what strides would be if computed FRESH for shape (3,2,4)
(i.e. as if it were a brand-new, contiguous tensor of this shape):

fresh stride[2] = 1
fresh stride[1] = 4
fresh stride[0] = 2*4 = 8

fresh strides = (8, 4, 1)

These two stride sets DIFFER: actual (4,12,1) vs fresh-expected (8,4,1).
This mismatch is the literal definition of "non-contiguous".

If something assumed the FRESH strides instead of the real ones
(exactly the mistake .view() would make if it were allowed to run):

t[1, 1, 0] using WRONG fresh strides -> RAM[8*1 + 4*1 + 1*0] = RAM[12] = 1b
                                          ✗ WRONG (expected 5b, got 1b)

This silent wrong mapping is exactly what .view() would risk producing,
which is why PyTorch makes .view() refuse to run on a non-contiguous
tensor rather than return corrupted data.

TO RESOLVE: .contiguous() copies the data into a NEW memory block,
laid out to match the CORRECT LOGICAL VIEW order:

[1a, 2a, 3a, 4a, 1b, 2b, 3b, 4b, 5a, 6a, 7a, 8a, 5b, 6b, 7b, 8b, 9a, 10a, 11a, 12a, 9b, 10b, 11b, 12b]

Now the fresh strides (8, 4, 1) genuinely match this new memory layout,
so .view() can safely relabel from here.
```

**In section 6** we applied `.view() -> .transpose()` so the `.view()` just restructured the **data representation**, but when `.tranpose()` the **data** itself was **reorganised**, and since `.view()` was never called for the `Attention Mechanism` this wasn't an issue. 

**In this section** we merge `.transpose() -> .view()` but this isn't possible without essentially matching the alignment between the RAM and the representation as such we `.transpose() -> .contiguous() -> .view()` where the data is relaid (in the RAM) in the current logical order, and then we can change the representation. 

### Style Note
For this reason some prefer to use `.reshape()` which esentially combines the `.contiguous()` when necessary; together with `.view()` avoiding any confusion. 

Others explicitely applies the above method to keep in the mind what's happen in memory.




In [46]:
v = torch.arange(0, 24, 1)
print(v)
print(v[11])
print('='*100)
v_reshaped = v.view(2, 3, 4)
print(v_reshaped)
print('='*100)
v_transposed = v_reshaped.transpose(0, 1) # shape (3, 2, 4), non-contiguous

print("TRUE or FALSE that v_transposed is contiguous: ", v_transposed.is_contiguous())

print(v_transposed)
# Uncomment below to trigger an error and then commment to move to the next section
# v_bad = v_transposed.view(2, 3, 4) # Triggers an error

print('='*100)
v_restored = v_transposed.contiguous()
v_good = v_restored.view(2, 3, 4)
print("The data was taken to another area of the RAM \n",v_good)




tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23])
tensor(11)
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
TRUE or FALSE that v_transposed is contiguous:  False
tensor([[[ 0,  1,  2,  3],
         [12, 13, 14, 15]],

        [[ 4,  5,  6,  7],
         [16, 17, 18, 19]],

        [[ 8,  9, 10, 11],
         [20, 21, 22, 23]]])
The data was taken to another area of the RAM 
 tensor([[[ 0,  1,  2,  3],
         [12, 13, 14, 15],
         [ 4,  5,  6,  7]],

        [[16, 17, 18, 19],
         [ 8,  9, 10, 11],
         [20, 21, 22, 23]]])


## Implementation

In this section implementing the full transformer can be written up without having multiple transformer blocks. 

In [47]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_heads):
        super().__init__()
        self.n_embed = n_embd
        self.n_heads = n_heads
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)
        self.ln_1 = nn.LayerNorm(n_embd)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.ln_f = nn.LayerNorm(n_embd)
        self.qkv = nn.Linear(n_embd, 3*n_embd)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd)
        )
        self.out_linear = nn.Linear(n_embd, vocab_size)

    def process_input(self, t):
        B, T = t.shape
        embd_token = self.wte(t)
        pos = torch.arange(T)
        pos_emb = self.wpe(pos)
        x_unnorm = embd_token + pos_emb
        x_norm = self.ln_1(x_unnorm)
        return x_norm

    def preprocess_multihead(self, t):
        B, T, C = t.shape
        t_reviewed = t.view(B, T, self.n_heads, C//self.n_heads)
        t_transposed = t_reviewed.transpose(1, 2)
        return t_transposed

    def attention_mech(self, q, k, v):
        T = q.shape[-2]
        head_dim = q.shape[-1]
        raw_score = q @ k.transpose(-1, -2)
        scaled_score = raw_score / (head_dim** 0.5)
        mask = torch.tril(torch.ones(T, T))
        masked_scores = scaled_score.masked_fill(mask==0, float('-inf')) 
        probs = masked_scores.softmax(dim=-1)
        return probs @ v

    def concat(self, t):
        t_trans = t.transpose(1, 2)
        B, T, _, _ = t_trans.shape
        t_reviewed = t_trans.contiguous().view(B, T, self.n_embed)
        return t_reviewed

    def forward(self, idx):
        # This is normalised and is used as part of the residual section 
        x = self.process_input(idx)

        qkv = self.qkv(x)
        q_raw, k_raw, v_raw = qkv.split(self.n_embed, dim=-1)

        q = self.preprocess_multihead(q_raw)
        k = self.preprocess_multihead(k_raw)
        v = self.preprocess_multihead(v_raw)

        weights = self.attention_mech(q, k, v)
        full_w = self.concat(weights)

        mixed_x = self.out_proj(full_w)

        # RESIDUAL 
        res_x_one = mixed_x + x

        norm_x_one = self.ln_2(res_x_one)
        # ff is for fed forwward, instead of feed
        x_ff = self.ff(norm_x_one)

        # RESIDUAL 
        res_x_two = x_ff + res_x_one

        norm_x_two = self.ln_f(res_x_two)
        final_weights = self.out_linear(norm_x_two)

        return final_weights

In [48]:
model = Transformer(vocab_size=6, n_embd=4, block_size=4, n_heads=2)
idx = torch.tensor([[0, 1, 2, 3]])
out = model(idx)
print(out.shape)

torch.Size([1, 4, 6])
